### Document Ingestion

In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is a test document.",
    metadata={
        "source": "test_source.txt",
        "author": "John Doe",
        }
)

In [3]:
doc

Document(metadata={'source': 'test_source.txt', 'author': 'John Doe'}, page_content='This is a test document.')

In [4]:
from langchain_community.document_loaders import TextLoader

f:\Programing Data\RAG\First_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
loader =(TextLoader("../data/ai_basics.txt",encoding="utf-8"))

In [6]:
data = loader.load()

In [7]:
data

[Document(metadata={'source': '../data/ai_basics.txt'}, page_content='Artificial Intelligence Basics\n\nArtificial Intelligence (AI) refers to the simulation of human\nintelligence in machines. Key areas include: - Machine Learning -\nNatural Language Processing - Computer Vision\n\nAI is widely used in recommendation systems, chatbots, and autonomous\nvehicles.\n')]

In [43]:
from langchain_community.document_loaders import DirectoryLoader
load = DirectoryLoader("../Data", glob="*.txt", show_progress=False, loader_kwargs={"encoding": "utf-8"}, loader_cls=TextLoader)
doc = load.load()

In [44]:
doc
len(doc)

3

In [45]:
from langchain_community.document_loaders import pdf

In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
chunker = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
chunks = chunker.split_documents(doc)




In [47]:
print(len(chunks))

13


[Document(metadata={'source': '..\\Data\\ai_basics.txt'}, page_content='Artificial Intelligence Basics'),
 Document(metadata={'source': '..\\Data\\ai_basics.txt'}, page_content='Artificial Intelligence (AI) refers to the simulation of human'),
 Document(metadata={'source': '..\\Data\\ai_basics.txt'}, page_content='intelligence in machines. Key areas include: - Machine Learning -'),
 Document(metadata={'source': '..\\Data\\ai_basics.txt'}, page_content='Natural Language Processing - Computer Vision'),
 Document(metadata={'source': '..\\Data\\ai_basics.txt'}, page_content='AI is widely used in recommendation systems, chatbots, and autonomous\nvehicles.'),
 Document(metadata={'source': '..\\Data\\database_and_faiss.txt'}, page_content='Database Systems and FAISS'),
 Document(metadata={'source': '..\\Data\\database_and_faiss.txt'}, page_content='A database is a structured collection of data. Types of databases: -'),
 Document(metadata={'source': '..\\Data\\database_and_faiss.txt'}, page_co

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


f:\Programing Data\RAG\First_rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ezio Auditore\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1388.03it/s]
BertModel LOAD

Model loaded successfully. Embedding dimension: 384


In [52]:
import os
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [53]:
chunks
text = [i.page_content for i in chunks]

['Artificial Intelligence Basics',
 'Artificial Intelligence (AI) refers to the simulation of human',
 'intelligence in machines. Key areas include: - Machine Learning -',
 'Natural Language Processing - Computer Vision',
 'AI is widely used in recommendation systems, chatbots, and autonomous\nvehicles.',
 'Database Systems and FAISS',
 'A database is a structured collection of data. Types of databases: -',
 'Relational (MySQL, PostgreSQL) - NoSQL (MongoDB, Redis)',
 'Vector databases like FAISS are used in AI applications such as RAG\nsystems.',
 'Web Development Overview',
 'Web development involves building websites and applications. It',
 'includes: - Frontend (HTML, CSS, JavaScript) - Backend (Python, Node.js,\ndatabases)',
 'Frameworks like React and FastAPI are commonly used.']

In [54]:
embeddings = embedding_manager.generate_embeddings(text)


Generating embeddings for 13 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]

Generated embeddings with shape: (13, 384)


In [55]:
vectorstore.add_documents(chunks, embeddings)

Adding 13 documents to vector store...
Successfully added 13 documents to vector store
Total documents in collection: 13


In [56]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [57]:
rag_retriever.retrieve("What is Artificial Intelligence?")

Retrieving documents for query: 'What is Artificial Intelligence?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.23it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_58f3c4a6_1',
  'content': 'Artificial Intelligence (AI) refers to the simulation of human',
  'metadata': {'source': '..\\Data\\ai_basics.txt',
   'doc_index': 1,
   'content_length': 62},
  'similarity_score': 0.5246398746967316,
  'distance': 0.47536012530326843,
  'rank': 1},
 {'id': 'doc_e8bf118b_0',
  'content': 'Artificial Intelligence Basics',
  'metadata': {'doc_index': 0,
   'source': '..\\Data\\ai_basics.txt',
   'content_length': 30},
  'similarity_score': 0.3964974880218506,
  'distance': 0.6035025119781494,
  'rank': 2},
 {'id': 'doc_2e4ddb2b_4',
  'content': 'AI is widely used in recommendation systems, chatbots, and autonomous\nvehicles.',
  'metadata': {'source': '..\\Data\\ai_basics.txt',
   'content_length': 79,
   'doc_index': 4},
  'similarity_score': 0.20872735977172852,
  'distance': 0.7912726402282715,
  'rank': 3},
 {'id': 'doc_1ba9b350_2',
  'content': 'intelligence in machines. Key areas include: - Machine Learning -',
  'metadata': {'content_leng